# Email RAG Pipeline: Interactive Email Parser

This notebook processes recruitment emails from your mailbox and structures them for the RAG system.

**Goal**: Extract and keep:
- sender
- date
- body
- signature (important!)
- person_name, company, job_title, contact_info

## Step 1: Import libraries and setup

In [63]:
import json
import re
import mailbox
from pathlib import Path
from email.parser import Parser as EmailParser
from typing import Dict, List, Optional
import pandas as pd

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Step 2: Define the EmailProcessor class

In [64]:
class EmailProcessor:
    """Process emails and extract structured fields for RAG system."""

    def __init__(self, input_path: str):
        self.input_path = Path(input_path)

    def parse(self) -> List[Dict]:
        """Parse mailbox and return structured email records."""
        records: List[Dict] = []

        if self.input_path.is_file():
            # Try to parse as mbox (works for .mbox files and bare mbox format files like "Inbox")
            records.extend(self._parse_mbox_file(self.input_path))
        elif self.input_path.is_dir():
            for file_path in self.input_path.glob("*.eml"):
                records.extend(self._parse_eml_file(file_path))
        return records

    def _parse_mbox_file(self, path: Path) -> List[Dict]:
        """Parse .mbox file and extract all emails."""
        records = []
        email_count = 0
        try:
            mbox = mailbox.mbox(str(path))
            for idx, msg in enumerate(mbox):
                email_count += 1
                record = self._extract_email_record(msg, idx)
                if record:
                    records.append(record)
            print(f"Found {email_count} emails in {path.name}")
        except Exception as e:
            print(f"Error parsing .mbox file: {e}")
        return records

    def _extract_email_record(self, msg, idx: int) -> Optional[Dict]:
        """Extract structured fields from one email."""
        try:
            sender = msg.get("From", "")
            date_str = msg.get("Date", "")
            # Format date to MM-DD format
            date = self._format_date(date_str)
            subject = msg.get("Subject", "")
            
            # Extract body
            body = ""
            if msg.is_multipart():
                for part in msg.walk():
                    if part.get_content_type() == "text/plain":
                        body = part.get_payload(decode=True).decode("utf-8", errors="ignore")
                        break
            else:
                body = msg.get_payload(decode=True).decode("utf-8", errors="ignore")
            
            # Extract signature and clean body
            signature = self.extract_signature(body)
            cleaned_body = self.remove_quoted_replies(body)
            
            # Extract key fields
            person_name = self.extract_person_name(signature)
            company = self.extract_company(signature)
            job_title = self.extract_job_title(signature)
            contact_info = self.extract_contact_info(signature, sender)
            
            return {
                "id": f"email_{idx}",
                "sender": sender,
                "date": date,
                "subject": subject,
                "body": cleaned_body.strip(),
                "signature": signature.strip(),
                "person_name": person_name,
                "company": company,
                "job_title": job_title,
                "contact_info": contact_info,
            }
        except Exception as e:
            print(f"Error extracting record: {e}")
            return None

    def _format_date(self, date_str: str) -> str:
        """Format email date to MM-DD format."""
        if not date_str:
            return ""
        try:
            from email.utils import parsedate_to_datetime
            dt = parsedate_to_datetime(date_str)
            return dt.strftime("%m-%d")
        except Exception:
            # If parsing fails, try simple regex extraction
            match = re.search(r"(\d{1,2})\s+(\w+)\s+\d{4}", date_str)
            if match:
                day = match.group(1).zfill(2)
                month_str = match.group(2)
                months = {"Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04",
                         "May": "05", "Jun": "06", "Jul": "07", "Aug": "08",
                         "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"}
                month = months.get(month_str, "")
                if month:
                    return f"{month}-{day}"
            return date_str

    def extract_signature(self, body: str) -> str:
        """Extract signature block from email body."""
        if "--" in body:
            parts = body.split("--")
            return parts[-1].strip()
        lines = body.strip().split("\n")
        if len(lines) > 3:
            return "\n".join(lines[-5:])
        return body.strip()

    def remove_quoted_replies(self, body: str) -> str:
        """Remove quoted replies (lines starting with '>'))."""
        lines = body.split("\n")
        cleaned = [line for line in lines if not line.strip().startswith(">")]
        return "\n".join(cleaned).strip()

    def extract_person_name(self, signature: str) -> str:
        """Extract person name from signature."""
        lines = signature.split("\n")
        for line in lines:
            line = line.strip()
            if line and "@" not in line and len(line) < 50:
                return line
        return ""

    def extract_company(self, signature: str) -> str:
        """Extract company name from signature."""
        patterns = [
            r"(?:at|from|@)\s+([A-Za-z0-9\s&\-\.]+?)(?:,|\n|$)",
        ]
        for pattern in patterns:
            match = re.search(pattern, signature, re.IGNORECASE)
            if match:
                company = match.group(1).strip()
                if len(company) < 100:
                    return company
        return ""

    def extract_job_title(self, signature: str) -> str:
        """Extract job title from signature."""
        patterns = [
            r"(?:Recruiter|Manager|Engineer|Developer|Designer|Lead|Head|Director|VP)",
        ]
        for pattern in patterns:
            match = re.search(pattern, signature, re.IGNORECASE)
            if match:
                return match.group(0)
        return ""

    def extract_contact_info(self, signature: str, sender: str) -> str:
        """Extract contact info (email, phone) from signature."""
        email_match = re.search(r"[\w\.-]+@[\w\.-]+\.\w+", signature)
        if email_match:
            return email_match.group(0)
        return sender

    def save_json(self, records: List[Dict], output_path: str) -> None:
        """Save records to JSON file."""
        output_dir = Path(output_path).parent
        output_dir.mkdir(parents=True, exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(records, f, ensure_ascii=False, indent=2)
        print(f"✓ Saved {len(records)} records to {output_path}")

print("✓ EmailProcessor class defined")

✓ EmailProcessor class defined


## Step 3: Load your mailbox

In [65]:
# Switch to use Inbox file instead
mbox_path = "../Takeout/Mail/Inbox"

# Check if file exists
if Path(mbox_path).exists():
    print(f"✓ Found mailbox: {mbox_path}")
else:
    print(f"✗ Mailbox not found: {mbox_path}")
    print("Please place your .mbox file in the correct location")

✓ Found mailbox: ../Takeout/Mail/Inbox


## Step 4: Process emails

In [66]:
# Initialize processor and parse emails
processor = EmailProcessor(mbox_path)
records = processor.parse()

print(f"\n✓ Successfully processed {len(records)} emails")

Found 172 emails in Inbox

✓ Successfully processed 172 emails


## Step 5: Preview the results

In [67]:
# Show a sample record
if records:
    print("Sample email record:")
    print(json.dumps(records[0], ensure_ascii=False, indent=2))

Sample email record:
{
  "id": "email_0",
  "sender": "Yahoo <no-reply@cc.yahoo.com>",
  "date": "06-28",
  "subject": "Sign in notification from Yahoo",
  "body": "<html>\n<head>\n    <meta charset=\"UTF-8\">\n    <style type=\"text/css\">\n        html {\n        -webkit-text-size-adjust:none;\n        }\n        body {\n        width:100%;\n        margin:0 auto;\n        padding:0;\n        }\n        li {\n        margin-top:15px;\n        }\n        @font-face {\n            font-family: 'YahooSans';\n            src: url(https://s.yimg.com/os/fontserver/YahooSans/Regular.woff2) format('woff2'),url(https://s.yimg.com/os/fontserver/YahooSans/Regular.woff) format('woff');\n            font-style: normal;\n            font-display: fallback\n        }\n\n        @font-face {\n            font-family: 'YahooSans';\n            src: url(https://s.yimg.com/os/fontserver/YahooSans/Medium.woff2) format('woff2'),url(https://s.yimg.com/os/fontserver/YahooSans/Medium.woff) format('woff');\n

## Step 6: Display as dataframe for quick review

In [68]:
# Convert to DataFrame for easier exploration
df = pd.DataFrame(records)

# Show key columns
display_cols = ["sender", "person_name", "company", "job_title", "contact_info", "date"]
print(f"\nEmail records summary ({len(df)} total):")
print(df[display_cols].head(10))


Email records summary (172 total):
                                              sender  \
0                      Yahoo <no-reply@cc.yahoo.com>   
1      Public Storage <DoNotReply@Publicstorage.com>   
2        Kristy Lariviere <klariviere@rmcagency.com>   
3        Kristy Lariviere <klariviere@rmcagency.com>   
4        Kristy Lariviere <klariviere@rmcagency.com>   
5      "venkat | rigelsky Inc" <venkat@rigelsky.com>   
6      "venkat | rigelsky Inc" <venkat@rigelsky.com>   
7  Rishu Upadhyay <rishu.upadhyay@tekinspirations...   
8                    "Janvi Jha" <janvi@vyzeinc.com>   
9                    "Janvi Jha" <janvi@vyzeinc.com>   

                                    person_name company job_title  \
0                                             >                     
1                                             |                     
2  www.rmcagency.com<http://www.rmcagency.com/>                     
3  www.rmcagency.com<http://www.rmcagency.com/>                     
4 

## Step 7: Save to JSON

In [69]:
# Save structured records for later use in RAG system
output_path = "./output/email_records.json"
processor.save_json(records, output_path)

✓ Saved 172 records to ./output/email_records.json


## Step 8: Inspect extracted fields

In [70]:
# Check what we extracted
print(f"\nExtracted field statistics:")
print(f"Emails with person_name: {df['person_name'].notna().sum()}")
print(f"Emails with company: {df['company'].notna().sum()}")
print(f"Emails with job_title: {df['job_title'].notna().sum()}")
print(f"Emails with contact_info: {df['contact_info'].notna().sum()}")

# Show top recruiters/companies
print(f"\nTop companies in your emails:")
print(df['company'].value_counts().head(10))


Extracted field statistics:
Emails with person_name: 172
Emails with company: 172
Emails with job_title: 172
Emails with contact_info: 172

Top companies in your emails:
company
                                                                                               133
any dissemination                                                                               16
should be unsubscribed. We apologize for any inconvenience                                       4
Trail                                                                                            3
collaboration and communication skills.                                                          3
should be unsubscribed. We apologize for any inconvenience.                                      3
this Zoom service allows audio and other information sent during the session to be recorded      2
sent you the invite for such purposes.                                                           2
sent you the invite.         

## Next steps

1. ✓ Parse and structure emails
2. → Create embeddings for each email
3. → Index into vector database (pgvector/Qdrant)
4. → Build retrieval pipeline
5. → Test RAG queries

## 第二部分: 向量嵌入 (Embeddings)

使用 sentence-transformers 为邮件内容生成向量表示

In [71]:
# Install sentence-transformers if needed
import subprocess
import sys

try:
    from sentence_transformers import SentenceTransformer
    print("✓ sentence-transformers already installed")
except ImportError:
    print("Installing sentence-transformers...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer
    print("✓ sentence-transformers installed")

✓ sentence-transformers already installed


In [78]:
# Load embedding model
print("Loading sentence-transformers model (this may take a moment)...")
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print(f"✓ Model loaded: {model.get_sentence_embedding_dimension()}-dimensional embeddings")

Loading sentence-transformers model (this may take a moment)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Model loaded: 384-dimensional embeddings


/tmp/ipykernel_35967/806121895.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✓ Model loaded: {model.get_sentence_embedding_dimension()}-dimensional embeddings")


In [79]:
# Generate embeddings for email content
# We'll embed: subject + body (most important content)
print(f"Generating embeddings for {len(records)} emails...")

embeddings_data = []
batch_size = 32

for i in range(0, len(records), batch_size):
    batch = records[i:i+batch_size]
    # Combine subject and body for embedding
    texts = [f"{record['subject']}\n{record['body']}" for record in batch]
    
    # Generate embeddings
    batch_embeddings = model.encode(texts, show_progress_bar=False)
    
    # Add to data
    for j, record in enumerate(batch):
        embeddings_data.append({
            "id": record["id"],
            "embedding": batch_embeddings[j].tolist()  # Convert numpy array to list
        })
    
    if (i + batch_size) % 100 == 0 or (i + batch_size) >= len(records):
        print(f"  Processed {min(i + batch_size, len(records))}/{len(records)} emails")

print(f"✓ Successfully generated {len(embeddings_data)} embeddings")

Generating embeddings for 172 emails...
  Processed 172/172 emails
✓ Successfully generated 172 embeddings


In [74]:
# Save embeddings to separate file
embeddings_path = Path("./output/email_embeddings.json")
embeddings_path.parent.mkdir(parents=True, exist_ok=True)

with open(embeddings_path, "w", encoding="utf-8") as f:
    json.dump(embeddings_data, f, ensure_ascii=False)

print(f"✓ Saved embeddings to {embeddings_path}")
print(f"  File size: {embeddings_path.stat().st_size / (1024*1024):.2f} MB")

✓ Saved embeddings to output/email_embeddings.json
  File size: 1.39 MB


In [80]:
# Verify embeddings
print("\\nEmbedding Statistics:")
print(f"Total embeddings: {len(embeddings_data)}")
print(f"Embedding dimension: {len(embeddings_data[0]['embedding'])}")
print(f"Sample embedding (first 5 dimensions): {embeddings_data[0]['embedding'][:5]}")
print(f"\\nEmbedding ID matches: {embeddings_data[0]['id']} == {records[0]['id']} ? {embeddings_data[0]['id'] == records[0]['id']}")

\nEmbedding Statistics:
Total embeddings: 172
Embedding dimension: 384
Sample embedding (first 5 dimensions): [-0.038591016083955765, 0.07147185504436493, -0.011270700953900814, 0.007569811772555113, 0.07745607197284698]
\nEmbedding ID matches: email_0 == email_0 ? True


## 第三部分: 数据库存储 (PostgreSQL + pgvector)

In [81]:
# Setup for database connection
import os
from dotenv import load_dotenv
import psycopg2
import psycopg2.extras

# Load environment variables from .env
load_dotenv()
NEON_PG_CONNECTION_URL = os.getenv('NEON_PG_CONNECTION_URL')

if not NEON_PG_CONNECTION_URL:
    print("⚠️  WARNING: NEON_PG_CONNECTION_URL not set in .env file")
    print("To use database storage:")
    print("1. Create a free account at https://console.neon.tech/")
    print("2. Get your connection string")
    print("3. Add it to /workspaces/Deep/.env as: NEON_PG_CONNECTION_URL=your_connection_string")
    NEON_PG_CONNECTION_URL = None
else:
    print(f"✓ NEON_PG_CONNECTION_URL found: {NEON_PG_CONNECTION_URL[:50]}...")

⚠️  WARNING: NEON_PG_CONNECTION_URL not set in .env file
To use database storage:
1. Create a free account at https://console.neon.tech/
2. Get your connection string
3. Add it to /workspaces/Deep/.env as: NEON_PG_CONNECTION_URL=your_connection_string


In [ ]:
# Connect to database and setup tables
connection = None
cursor = None

if NEON_PG_CONNECTION_URL:
    try:
        connection = psycopg2.connect(NEON_PG_CONNECTION_URL)
        connection.autocommit = True
        cursor = connection.cursor()
        print("✓ Connected to Neon PostgreSQL!")
    except Exception as e:
        print(f"✗ Cannot connect to Neon PostgreSQL: {e}")
        NEON_PG_CONNECTION_URL = None
else:
    print("ℹ️  Skipping database setup (no connection string provided)")

In [ ]:
# Create pgvector extension and table
if cursor:
    try:
        cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        print("✓ Created pgvector extension")
    except Exception as e:
        print(f"⚠️  pgvector extension error: {e}")

    try:
        cursor.execute("DROP TABLE IF EXISTS emails_rag;")
        print("✓ Dropped existing table")
    except Exception as e:
        print(f"Note: {e}")

    try:
        cursor.execute("""
            CREATE TABLE emails_rag (
                id BIGSERIAL PRIMARY KEY,
                email_id TEXT UNIQUE,
                sender TEXT,
                email_date VARCHAR(10),
                subject TEXT,
                body TEXT,
                signature TEXT,
                person_name TEXT,
                company TEXT,
                job_title TEXT,
                contact_info TEXT,
                embedding VECTOR(384)
            );
        """)
        print("✓ Created table emails_rag")
    except Exception as e:
        print(f"✗ Cannot create table: {e}")

In [ ]:
# Insert emails and embeddings into database
if cursor and len(records) == len(embeddings_data):
    print(f"Inserting {len(records)} emails into database...")
    
    try:
        for i, record in enumerate(records):
            embedding = embeddings_data[i]['embedding']
            
            cursor.execute("""
                INSERT INTO emails_rag 
                (email_id, sender, email_date, subject, body, signature, 
                 person_name, company, job_title, contact_info, embedding)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (email_id) DO NOTHING;
            """, (
                record['id'],
                record['sender'],
                record['date'],
                record['subject'],
                record['body'],
                record['signature'],
                record['person_name'],
                record['company'],
                record['job_title'],
                record['contact_info'],
                embedding
            ))
        
        print(f"✓ Successfully inserted {len(records)} emails")
    except Exception as e:
        print(f"✗ Error inserting emails: {e}")
elif cursor:
    print(f"⚠️  Mismatch: {len(records)} records vs {len(embeddings_data)} embeddings")
else:
    print("ℹ️  Database not connected - skipping insertion")

In [ ]:
# Verify database insertion
if cursor:
    try:
        cursor.execute("SELECT COUNT(*) FROM emails_rag;")
        count = cursor.fetchone()[0]
        print(f"✓ Database verification: {count} emails stored")
        
        # Show sample record
        cursor.execute("""
            SELECT email_id, subject, company, person_name 
            FROM emails_rag LIMIT 1;
        """)
        sample = cursor.fetchone()
        if sample:
            print(f"\nSample record from database:")
            print(f"  ID: {sample[0]}")
            print(f"  Subject: {sample[1][:60]}...")
            print(f"  Company: {sample[2]}")
            print(f"  Person: {sample[3]}")
    except Exception as e:
        print(f"Error verifying database: {e}")
else:
    print("Database not available for verification")